## Exercício Prático 3

Nome do Aluno 1: Anderson Cardoso  
N°Pece: 119896

# Exercício: Classificação de Espécies de Flores com XGBoost

Neste exercício, você usará a famosa **base de dados Iris** para treinar e avaliar modelos de **classificação com métodos ensemble, incluindo o XGBoost**.

A base contém 150 amostras de três espécies de flores (*Iris setosa*, *Iris versicolor*, *Iris virginica*), com as seguintes variáveis:

- Comprimento da sépala (`sepal length`)
- Largura da sépala (`sepal width`)
- Comprimento da pétala (`petal length`)
- Largura da pétala (`petal width`)




## Objetivo

Utilizar **métodos ensemble, incluindo o XGBoost** para **prever a espécie da flor** com base em suas características morfológicas.

---

## O que você deve fazer:

OBS: Aproveite o código do exercício da aula anterior!

1. **Importar a base Iris** usando o código fornecido.
2. Separar os dados entre variáveis preditoras (`X`) e alvo (`y`).
3. Dividir os dados em treino e teste (por exemplo, 70%/30%).
4. Utilizar os métodos **XGBoost**, Bagging, Random Forest, AdaBoost e Gradient Boosting com os dados de treino.
5. Avaliar o desempenho dos métodos.


A base e as instruções para importação estão disponíveis em:

https://archive.ics.uci.edu/dataset/53/iris

# Insira o seu código aqui

In [9]:
# Importa os modelos
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from ucimlrepo import fetch_ucirepo 
import matplotlib.pyplot as plt

# fetch dataset 
iris = fetch_ucirepo(id=53) 

# data (as pandas dataframes) 
X = iris.data.features
y = iris.data.targets

# Treino e teste divisão
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42, # fixar ponto de partida para reprodutibilidade
    stratify=y # garante que a proporção de classes seja mantida em ambos os conjuntos de treino e teste

)
# Codifica variáveis categóricas ('Sim' → 1, 'Nao' → 0)
le = LabelEncoder()
y_train = le.fit_transform(y_train.values.ravel())
y_test = le.transform(y_test.values.ravel())
  

In [11]:
# Modelos Ensemble
modelos = {
    "XGBoost": XGBClassifier,
    "Bagging": BaggingClassifier,
    "Random Forest": RandomForestClassifier,
    "AdaBoost": AdaBoostClassifier,
    "Gradient Boosting": GradientBoostingClassifier
}

melhores_modelos = {}

for nome, classe_modelo in modelos.items():
    n = 10

    melhor_accuracy = 0
    melhor_modelo = None
    melhor_n = 0
    melhor_y_pred = None

    while n <= 500:
        if nome == "Bagging":
            modelo = classe_modelo(
                estimator=DecisionTreeClassifier(random_state=42),
                n_estimators=n,
                random_state=42
            )
        elif nome == "XGBoost":
            modelo = classe_modelo(
                n_estimators=n,            
                learning_rate=0.1,         # taxa de aprendizado
                max_depth=3,               # profundidade máxima das árvores
                random_state=42,           
            )
        else:
            modelo = classe_modelo(
                n_estimators=n,
                random_state=42
            )
            

        # Treinar
        modelo.fit(X_train, y_train)

        # Prever
        y_pred = modelo.predict(X_test)

        # Accuracy
        accuracy = accuracy_score(y_test, y_pred)

        # Guardar melhor resultado
        if accuracy > melhor_accuracy:
            melhor_accuracy = accuracy
            melhor_modelo = modelo
            melhor_n = n
            melhor_y_pred = y_pred

        n += 10

    # Guarda tudo do melhor modelo
    melhores_modelos[nome] = {
        "modelo": melhor_modelo,
        "n_estimators": melhor_n,
        "accuracy": melhor_accuracy,
        "report": classification_report(
            y_test,
            melhor_y_pred
        ),
        "matrix": confusion_matrix(
            y_test,
            melhor_y_pred
        )
    }

In [ ]:
# XGBoost Resultado
resultado = melhores_modelos["XGBoost"]
print("N_estimators:", resultado["n_estimators"])
print("Accuracy:", resultado["accuracy"])

print("\nClassification Report:")
print(resultado["report"])

print("\nConfusion Matrix:")
print(resultado["matrix"])

N_estimators: 10
Accuracy: 0.9555555555555556

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.93      0.93      0.93        15
           2       0.93      0.93      0.93        15

    accuracy                           0.96        45
   macro avg       0.96      0.96      0.96        45
weighted avg       0.96      0.96      0.96        45


Confusion Matrix:
[[15  0  0]
 [ 0 14  1]
 [ 0  1 14]]


In [12]:
# Bagging Resultado
resultado = melhores_modelos["Bagging"]
print("N_estimators:", resultado["n_estimators"])
print("Accuracy:", resultado["accuracy"])

print("\nClassification Report:")
print(resultado["report"])

print("\nConfusion Matrix:")
print(resultado["matrix"])

N_estimators: 10
Accuracy: 0.9333333333333333

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.88      0.93      0.90        15
           2       0.93      0.87      0.90        15

    accuracy                           0.93        45
   macro avg       0.93      0.93      0.93        45
weighted avg       0.93      0.93      0.93        45


Confusion Matrix:
[[15  0  0]
 [ 0 14  1]
 [ 0  2 13]]


In [13]:
# Random Forest Resultado
resultado = melhores_modelos["Random Forest"]
print("N_estimators:", resultado["n_estimators"])
print("Accuracy:", resultado["accuracy"])

print("\nClassification Report:")
print(resultado["report"])

print("\nConfusion Matrix:")
print(resultado["matrix"])

N_estimators: 10
Accuracy: 0.9111111111111111

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.82      0.93      0.88        15
           2       0.92      0.80      0.86        15

    accuracy                           0.91        45
   macro avg       0.92      0.91      0.91        45
weighted avg       0.92      0.91      0.91        45


Confusion Matrix:
[[15  0  0]
 [ 0 14  1]
 [ 0  3 12]]


In [14]:
# AdaBoost Resultado
resultado = melhores_modelos["AdaBoost"]
print("N_estimators:", resultado["n_estimators"])
print("Accuracy:", resultado["accuracy"])

print("\nClassification Report:")
print(resultado["report"])

print("\nConfusion Matrix:")
print(resultado["matrix"])

N_estimators: 20
Accuracy: 0.9555555555555556

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.93      0.93      0.93        15
           2       0.93      0.93      0.93        15

    accuracy                           0.96        45
   macro avg       0.96      0.96      0.96        45
weighted avg       0.96      0.96      0.96        45


Confusion Matrix:
[[15  0  0]
 [ 0 14  1]
 [ 0  1 14]]


In [15]:
# Gradient Boosting Resultado
resultado = melhores_modelos["Gradient Boosting"]
print("N_estimators:", resultado["n_estimators"])
print("Accuracy:", resultado["accuracy"])

print("\nClassification Report:")
print(resultado["report"])

print("\nConfusion Matrix:")
print(resultado["matrix"])

N_estimators: 10
Accuracy: 0.9777777777777777

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      0.93      0.97        15
           2       0.94      1.00      0.97        15

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45


Confusion Matrix:
[[15  0  0]
 [ 0 14  1]
 [ 0  0 15]]


In [17]:
# Comparação final
for nome, resultado in melhores_modelos.items():
    print(
        f"{nome:<20} "
        f"Accuracy: {resultado['accuracy']:.4f} "
        f"| n_estimators: {resultado['n_estimators']}"
    )

XGBoost              Accuracy: 0.9556 | n_estimators: 10
Bagging              Accuracy: 0.9333 | n_estimators: 10
Random Forest        Accuracy: 0.9111 | n_estimators: 10
AdaBoost             Accuracy: 0.9556 | n_estimators: 20
Gradient Boosting    Accuracy: 0.9778 | n_estimators: 10
